In [32]:
import pandas as pd
import numpy as np
RAW = "../data/raw/"

results      = pd.read_csv(RAW + "results.csv")
races        = pd.read_csv(RAW + "races.csv")
drivers      = pd.read_csv(RAW + "drivers.csv")
constructors = pd.read_csv(RAW + "constructors.csv")
circuits     = pd.read_csv(RAW + "circuits.csv")
status       = pd.read_csv(RAW + "status.csv")

In [33]:
#First we read all the raw data and find out where/how to join the tables. 

results.info()
results.head()

races.info()
races.head()

drivers.info()
drivers.head()

constructors.info()
constructors.head()

circuits.info()
circuits.head()

status.info()
status.head()



<class 'pandas.DataFrame'>
RangeIndex: 26291 entries, 0 to 26290
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   resultId         26291 non-null  int64  
 1   raceId           26291 non-null  int64  
 2   driverId         26291 non-null  int64  
 3   constructorId    26291 non-null  int64  
 4   number           26285 non-null  float64
 5   grid             26291 non-null  int64  
 6   position         16574 non-null  float64
 7   positionText     26291 non-null  str    
 8   positionOrder    26291 non-null  int64  
 9   points           26291 non-null  float64
 10  laps             26291 non-null  int64  
 11  time             8631 non-null   str    
 12  milliseconds     8667 non-null   float64
 13  fastestLap       9130 non-null   float64
 14  rank             9154 non-null   float64
 15  fastestLapTime   9130 non-null   str    
 16  fastestLapSpeed  8238 non-null   float64
 17  statusId         26291 

,statusId,status
0,1,Finished
1,2,Disqualified
2,3,Accident
3,4,Collision
4,5,Engine


- results.raceId -> races (year, race name, circuitId)
- results.driverId -> drivers (driver name, nationality)
- results.constructorId -> constructors (team name)
- results.statusId -> status (finish or retirement reason)
- races.circuitId -> circuits (track name, country)

- resultId, raceId, driverId, constructorId, statusId, circuitId


In [35]:
#Keep only the columns we need; rename "name" so they don't clash

races_small = races[["raceId", "year", "name", "circuitId"]].rename(columns={"name": "race_name"})
drivers_small = drivers[["driverId", "forename", "surname", "nationality"]]
constructors_small = constructors[["constructorId", "name"]].rename(columns={"name": "team"})
circuits_small = circuits[["circuitId", "name", "country"]].rename(columns={"name": "circuit"})

In [36]:
#Merge everything onto results

df = results.merge(races_small, on="raceId", how="left")
df = df.merge(drivers_small, on="driverId", how="left")
df = df.merge(constructors_small, on="constructorId", how="left")
df = df.merge(status, on="statusId", how="left")
df = df.merge(circuits_small, on="circuitId", how="left")

len(results), len(df)

(26291, 26291)

In [37]:
#Output of results (with all the merged data)

df[["year", "race_name", "forename", "surname", "team", "grid", "position", "status", "circuit"]].head(10)

,year,race_name,forename,surname,team,grid,position,status,circuit
0,1950,British Grand Prix,Nino,Farina,Alfa Romeo,1,1.0,Finished,Silverstone Circuit
1,1950,British Grand Prix,Luigi,Fagioli,Alfa Romeo,2,2.0,Finished,Silverstone Circuit
2,1950,British Grand Prix,Reg,Parnell,Alfa Romeo,4,3.0,Finished,Silverstone Circuit
3,1950,British Grand Prix,Yves,Cabantous,Talbot-Lago,6,4.0,+2 Laps,Silverstone Circuit
4,1950,British Grand Prix,Louis,Rosier,Talbot-Lago,9,5.0,+2 Laps,Silverstone Circuit
5,1950,British Grand Prix,Bob,Gerard,ERA,13,6.0,+3 Laps,Silverstone Circuit
6,1950,British Grand Prix,Cuth,Harrison,ERA,15,7.0,+3 Laps,Silverstone Circuit
7,1950,British Grand Prix,Philippe,Étancelin,Talbot-Lago,14,8.0,+5 Laps,Silverstone Circuit
8,1950,British Grand Prix,David,Hampshire,Maserati,16,9.0,+6 Laps,Silverstone Circuit
9,1950,British Grand Prix,Joe,Fry,Maserati,20,10.0,+6 Laps,Silverstone Circuit


In [38]:
df["decade"] = (df["year"] // 10) * 10                # 1957 -> 1950 to get the decade
df["driver"] = df["forename"] + " " + df["surname"]    # full name for charts
df["pole"] = df["grid"] == 1                           # started first?
df["won"] = df["position"] == 1                        # finished first?
df["finished"] = df["position"].notna()                # has a finishing position?

In [39]:
df.to_csv("../data/processed/results_joined.csv", index=False)